# SPCS Internal Path: JDBC & ADBC Connection Tests

**Goal:** Verify that both JDBC and ADBC can connect to Snowflake from
Workspace Notebooks using the SPCS OAuth token and the internal host
(`SNOWFLAKE_HOST`), then benchmark write performance.

**Background:** The `spcs_snowdriver_lib` examples show that *all* Snowflake
drivers (Go, Java, Python, Node.js, .NET) can connect via SPCS OAuth when
the token is passed as a property/parameter (not embedded in a URL).

**Key insight:** Our previous JDBC test failed because the OAuth token was
embedded in the JDBC URL query string, where `+`, `/`, `=` characters got
mangled by URL parsing. The fix: pass the token via Java `Properties`.
For ADBC, use `auth_oauth` (not `auth_pat`) + `SNOWFLAKE_HOST`.

**Sections:**
1. Prerequisites (R env, Scala/Java env, rJava/RJDBC)
2. Session config & environment variables
3. JDBC connection test (Properties-based, fixed)
4. ADBC connection test (OAuth + internal host)
5. Write benchmarks (JDBC, ADBC, literal)
6. Results comparison

## 1. Prerequisites

Run the R environment setup and Scala/Java setup if not already done
in this session.

In [ ]:
# Install R environment with ADBC and register %%R magic
from sfnb_setup import setup_notebook
setup_notebook(config="rsnowflake_config.yaml", packages=["RSnowflake"])

In [ ]:
import subprocess, os, json

MAMBA = os.path.expanduser("~/micromamba/bin/micromamba")
R_ENV = "r_env"

def mamba_install(pkgs):
    cmd = [MAMBA, "install", "-n", R_ENV, "-c", "conda-forge", "-y", "--quiet"] + pkgs
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"FAILED: {r.stderr[-500:]}")
    else:
        print(f"Installed: {', '.join(pkgs)}")

mamba_install(["r-rjava", "openjdk"])
mamba_install(["r-rjdbc"])

## 2. Session Config & Environment Variables

In [ ]:
import os, glob
from snowflake.snowpark.context import get_active_session

session = get_active_session()

os.environ["SNOWFLAKE_ACCOUNT"]   = session.get_current_account().replace('"', '')
os.environ["SNOWFLAKE_USER"]      = session.sql("SELECT CURRENT_USER()").collect()[0][0]
os.environ["SNOWFLAKE_DATABASE"]  = (session.get_current_database() or "").replace('"', '')
os.environ["SNOWFLAKE_SCHEMA"]    = (session.get_current_schema() or "").replace('"', '')
os.environ["SNOWFLAKE_WAREHOUSE"] = (session.get_current_warehouse() or "").replace('"', '')
os.environ["SNOWFLAKE_ROLE"]      = (session.get_current_role() or "").replace('"', '')

for k in ["SNOWFLAKE_ACCOUNT", "SNOWFLAKE_HOST", "SNOWFLAKE_DATABASE",
          "SNOWFLAKE_SCHEMA", "SNOWFLAKE_WAREHOUSE", "SNOWFLAKE_ROLE"]:
    print(f"{k}: {os.environ.get(k, '<not set>')}")

# Locate JDBC JAR
meta_path = os.path.expanduser("~/scala_jars/scala_metadata.json")
jdbc_jar = None
if os.path.isfile(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    cp_file = meta.get("snowpark_classpath_file", "")
    if cp_file and os.path.isfile(cp_file):
        with open(cp_file) as f:
            jars = f.read().strip().split(":")
        jdbc_jars = [j for j in jars if "snowflake-jdbc" in j and j.endswith(".jar")]
        if jdbc_jars:
            jdbc_jar = jdbc_jars[0]
if not jdbc_jar:
    for pattern in [
        os.path.expanduser("~/scala_jars/**/snowflake-jdbc-*.jar"),
        os.path.expanduser("~/.cache/coursier/**/snowflake-jdbc-*.jar"),
    ]:
        candidates = glob.glob(pattern, recursive=True)
        if candidates:
            jdbc_jar = sorted(candidates)[-1]
            break
if jdbc_jar:
    os.environ["SNOWFLAKE_JDBC_JAR"] = jdbc_jar
    print(f"JDBC JAR: {jdbc_jar}")
else:
    print("WARNING: JDBC JAR not found. Run setup_scala_environment.sh first.")
    print("JDBC tests will be skipped.")

## 3. JDBC Connection Test (Properties-based)

**Fix:** Pass the OAuth token via Java `Properties` object, NOT in the URL.
The previous test embedded the token in the URL query string, where `+`,
`/`, `=` characters got mangled by URL parsing, causing "Invalid OAuth
access token" errors.

This matches the pattern in `spcs_snowdriver_lib/java/ConnectionFactory.java`.

In [ ]:
%%R
library(rJava)
library(DBI)
library(RJDBC)

# --- Locate the JDBC JAR ---
jdbc_jar <- Sys.getenv("SNOWFLAKE_JDBC_JAR", "")
if (!nzchar(jdbc_jar) || !file.exists(jdbc_jar)) {
  cat("JDBC JAR not found -- skipping JDBC test.\n")
  .GlobalEnv$jdbc_ok <- FALSE
} else {
  cat("JDBC JAR:", jdbc_jar, "\n")

  sf_driver <- JDBC(
    driverClass = "net.snowflake.client.jdbc.SnowflakeDriver",
    classPath   = jdbc_jar
  )
  cat("Driver loaded OK\n")

  # --- Read SPCS OAuth token ---
  token_path <- "/snowflake/session/token"
  if (!file.exists(token_path)) {
    stop("Not running in Workspace (no SPCS token)")
  }
  token <- trimws(readLines(token_path, warn = FALSE))
  cat("SPCS token loaded (length:", nchar(token), ")\n")

  # --- Use SNOWFLAKE_HOST (internal SPCS gateway) ---
  host <- Sys.getenv("SNOWFLAKE_HOST", "")
  if (!nzchar(host)) {
    host <- paste0(Sys.getenv("SNOWFLAKE_ACCOUNT"), ".snowflakecomputing.com")
    cat("WARNING: SNOWFLAKE_HOST not set, using public endpoint\n")
  }
  cat("Host:", host, "\n")

  # --- FIXED: Pass token via Properties, NOT in URL ---
  # Minimal URL -- only the host, no query params
  jdbc_url <- sprintf("jdbc:snowflake://%s/", host)

  tryCatch({
    jdbc_con <- dbConnect(
      sf_driver,
      jdbc_url,
      authenticator = "oauth",
      token         = token,
      account       = Sys.getenv("SNOWFLAKE_ACCOUNT"),
      db            = Sys.getenv("SNOWFLAKE_DATABASE"),
      schema        = Sys.getenv("SNOWFLAKE_SCHEMA"),
      warehouse     = Sys.getenv("SNOWFLAKE_WAREHOUSE"),
      role          = Sys.getenv("SNOWFLAKE_ROLE")
    )

    res <- dbGetQuery(jdbc_con, "SELECT CURRENT_ACCOUNT() AS acct, CURRENT_USER() AS usr, CURRENT_ROLE() AS role")
    cat("\n=== JDBC CONNECTION SUCCEEDED ===\n")
    cat(sprintf("  Account: %s, User: %s, Role: %s\n", res$ACCT, res$USR, res$ROLE))
    cat("  Token passed via Properties -- no URL encoding issues.\n")
    cat("  Using internal SPCS host:", host, "\n")

    .GlobalEnv$jdbc_con  <- jdbc_con
    .GlobalEnv$sf_driver <- sf_driver
    .GlobalEnv$jdbc_ok   <- TRUE

  }, error = function(e) {
    cat("\n=== JDBC CONNECTION FAILED ===\n")
    cat("Error:", conditionMessage(e), "\n")
    .GlobalEnv$jdbc_ok <- FALSE
  })
}

## 4. ADBC Connection Test (OAuth + Internal Host)

**Fix:** Use `auth_oauth` (not `auth_pat`) and `SNOWFLAKE_HOST` (not
the public endpoint). The ADBC Snowflake driver wraps `gosnowflake`,
which is shown to work with SPCS OAuth in `spcs_snowdriver_lib/go/`.

ADBC options:
- `adbc.snowflake.sql.auth_type` = `auth_oauth`
- `adbc.snowflake.sql.client_option.auth_token` = SPCS session token
- `adbc.snowflake.sql.uri.host` = `SNOWFLAKE_HOST` (internal)

In [ ]:
%%R
has_adbc <- requireNamespace("adbcsnowflake", quietly = TRUE) &&
            requireNamespace("adbcdrivermanager", quietly = TRUE)

if (!has_adbc) {
  cat("ADBC packages not installed -- skipping ADBC test.\n")
  cat("Run: setup_notebook() with ADBC config enabled\n")
  .GlobalEnv$adbc_ok <- FALSE
} else {
  cat("ADBC packages available\n")

  # --- Read SPCS OAuth token ---
  token_path <- "/snowflake/session/token"
  if (!file.exists(token_path)) {
    stop("Not running in Workspace (no SPCS token)")
  }
  token <- trimws(readLines(token_path, warn = FALSE))

  # --- Use SNOWFLAKE_HOST (internal) ---
  host <- Sys.getenv("SNOWFLAKE_HOST", "")
  if (!nzchar(host)) {
    host <- paste0(Sys.getenv("SNOWFLAKE_ACCOUNT"), ".snowflakecomputing.com")
    cat("WARNING: SNOWFLAKE_HOST not set, falling back to public endpoint\n")
  }
  cat("Host:", host, "\n")

  tryCatch({
    db <- adbcdrivermanager::adbc_database_init(
      adbcsnowflake::adbcsnowflake(),
      `adbc.snowflake.sql.account`   = Sys.getenv("SNOWFLAKE_ACCOUNT"),
      `adbc.snowflake.sql.uri.host`  = host,
      `adbc.snowflake.sql.db`        = Sys.getenv("SNOWFLAKE_DATABASE"),
      `adbc.snowflake.sql.schema`    = Sys.getenv("SNOWFLAKE_SCHEMA"),
      `adbc.snowflake.sql.warehouse` = Sys.getenv("SNOWFLAKE_WAREHOUSE"),
      `adbc.snowflake.sql.role`      = Sys.getenv("SNOWFLAKE_ROLE"),
      `adbc.snowflake.sql.auth_type` = "auth_oauth",
      `adbc.snowflake.sql.client_option.auth_token` = token
    )
    con <- adbcdrivermanager::adbc_connection_init(db)

    # Test query
    stmt <- adbcdrivermanager::adbc_statement_init(con)
    adbcdrivermanager::adbc_statement_set_sql_query(
      stmt, "SELECT CURRENT_ACCOUNT() AS acct, CURRENT_USER() AS usr"
    )
    stream <- adbcdrivermanager::adbc_statement_execute_query(stmt)[[1]]
    res <- as.data.frame(stream)

    cat("\n=== ADBC CONNECTION SUCCEEDED (OAuth + internal host) ===\n")
    cat(sprintf("  Account: %s, User: %s\n", res$ACCT, res$USR))
    cat("  auth_type: auth_oauth\n")
    cat("  host:", host, "\n")

    .GlobalEnv$adbc_db  <- db
    .GlobalEnv$adbc_con <- con
    .GlobalEnv$adbc_ok  <- TRUE

  }, error = function(e) {
    cat("\n=== ADBC CONNECTION FAILED ===\n")
    cat("Error:", conditionMessage(e), "\n\n")
    cat("If OAuth is rejected, try auth_pat + public endpoint as fallback.\n")
    .GlobalEnv$adbc_ok <- FALSE
  })
}

## 5. Write Benchmarks

For each available connection method, benchmark PUT + COPY INTO writes
for 5K and 50K rows. All methods use the same test data.

In [ ]:
%%R
cat("=== Write Benchmarks ===\n\n")

# -- Helper: JDBC write via PUT + COPY INTO --
jdbc_write_bench <- function(con, df, tbl_name) {
  tmp_csv <- tempfile(fileext = ".csv.gz")
  on.exit(unlink(tmp_csv), add = TRUE)
  gz <- gzfile(tmp_csv, "w")
  write.csv(df, gz, row.names = FALSE)
  close(gz)

  dbExecute(con, sprintf("DROP TABLE IF EXISTS %s", tbl_name))
  dbExecute(con, sprintf(
    "CREATE TABLE %s (ID NUMBER, TXT VARCHAR, VAL FLOAT)", tbl_name
  ))

  t0 <- proc.time()
  dbExecute(con, sprintf(
    "PUT 'file://%s' @%%25%s AUTO_COMPRESS=FALSE OVERWRITE=TRUE",
    tmp_csv, tbl_name
  ))
  dbExecute(con, sprintf(
    "COPY INTO %s FROM @%%25%s FILE_FORMAT=(TYPE=CSV SKIP_HEADER=1 FIELD_OPTIONALLY_ENCLOSED_BY='\"' COMPRESSION=GZIP) PURGE=TRUE",
    tbl_name, tbl_name
  ))
  elapsed <- (proc.time() - t0)[["elapsed"]]

  cnt <- dbGetQuery(con, sprintf("SELECT COUNT(*) AS N FROM %s", tbl_name))$N
  dbExecute(con, sprintf("DROP TABLE IF EXISTS %s", tbl_name))
  list(elapsed = elapsed, rows = cnt)
}

# -- Helper: ADBC bulk ingest --
adbc_write_bench <- function(db, con, df, tbl_name) {
  # Drop if exists
  stmt_drop <- adbcdrivermanager::adbc_statement_init(con)
  tryCatch({
    adbcdrivermanager::adbc_statement_set_sql_query(
      stmt_drop, sprintf("DROP TABLE IF EXISTS %s", tbl_name)
    )
    adbcdrivermanager::adbc_statement_execute_query(stmt_drop)
  }, error = function(e) NULL)

  t0 <- proc.time()
  stmt <- adbcdrivermanager::adbc_statement_init(
    con, `adbc.ingest.target_table` = tbl_name,
    `adbc.ingest.mode` = "adbc.ingest.mode.create"
  )
  adbcdrivermanager::adbc_statement_bind_stream(stmt, nanoarrow::as_nanoarrow_array_stream(df))
  adbcdrivermanager::adbc_statement_execute_query(stmt)
  elapsed <- (proc.time() - t0)[["elapsed"]]

  # Verify
  stmt_cnt <- adbcdrivermanager::adbc_statement_init(con)
  adbcdrivermanager::adbc_statement_set_sql_query(
    stmt_cnt, sprintf("SELECT COUNT(*) AS N FROM %s", tbl_name)
  )
  cnt_stream <- adbcdrivermanager::adbc_statement_execute_query(stmt_cnt)[[1]]
  cnt <- as.data.frame(cnt_stream)$N

  # Cleanup
  stmt_cl <- adbcdrivermanager::adbc_statement_init(con)
  adbcdrivermanager::adbc_statement_set_sql_query(
    stmt_cl, sprintf("DROP TABLE IF EXISTS %s", tbl_name)
  )
  adbcdrivermanager::adbc_statement_execute_query(stmt_cl)

  list(elapsed = elapsed, rows = cnt)
}

# -- Run benchmarks --
results <- list()

for (n in c(5000L, 50000L)) {
  label <- paste0(n / 1000, "K")
  cat(sprintf("--- %s rows ---\n", label))

  df <- data.frame(
    ID  = seq_len(n),
    TXT = paste0("row_", seq_len(n)),
    VAL = runif(n),
    stringsAsFactors = FALSE
  )

  # JDBC write
  if (exists("jdbc_ok") && jdbc_ok) {
    tryCatch({
      r <- jdbc_write_bench(jdbc_con, df, paste0("BENCH_JDBC_", label))
      cat(sprintf("  JDBC:    %6.2fs  (rows: %d)\n", r$elapsed, r$rows))
      results[[paste0("jdbc_", label)]] <- r$elapsed
    }, error = function(e) {
      cat(sprintf("  JDBC:    FAILED -- %s\n", conditionMessage(e)))
    })
  } else {
    cat("  JDBC:    skipped (not connected)\n")
  }

  # ADBC write (bulk ingest)
  if (exists("adbc_ok") && adbc_ok) {
    tryCatch({
      r <- adbc_write_bench(adbc_db, adbc_con, df, paste0("BENCH_ADBC_", label))
      cat(sprintf("  ADBC:    %6.2fs  (rows: %d)\n", r$elapsed, r$rows))
      results[[paste0("adbc_", label)]] <- r$elapsed
    }, error = function(e) {
      cat(sprintf("  ADBC:    FAILED -- %s\n", conditionMessage(e)))
    })
  } else {
    cat("  ADBC:    skipped (not connected)\n")
  }

  cat("\n")
}

## 6. Results Comparison

Fill in from above and previous tests:

| Method | 5K rows | 50K rows | Path | Auth |
| --- | --- | --- | --- | --- |
| Py `write_pandas` | 2.49s | 2.57s | PUT+COPY, internal SPCS | SPCS OAuth |
| Py literal INSERT | 1.03s | 10.62s | SQL VALUES, internal | SPCS OAuth |
| R literal INSERT | ~3s | ~45s | SQL VALUES, SQL API v2 | session token |
| R ADBC (old: PAT+public) | ~25s | ~21s | PUT+COPY, public | PAT |
| **R JDBC (Properties)** | **???** | **???** | **PUT+COPY, internal?** | **SPCS OAuth** |
| **R ADBC (OAuth+internal)** | **???** | **???** | **PUT+COPY, internal?** | **SPCS OAuth** |
| R -> rpy2 -> write_pandas | ~2.7s | ~2.8s | rpy2 + internal SPCS | SPCS OAuth |

If JDBC and/or ADBC with OAuth achieve ~2-3s via the internal path, they
are the cleanest solutions (pure R, no Python dependency for writes).

In [ ]:
%%R
# Cleanup connections
if (exists("jdbc_con") && inherits(jdbc_con, "JDBCConnection")) {
  tryCatch(dbDisconnect(jdbc_con), error = function(e) NULL)
  cat("JDBC connection closed.\n")
}
if (exists("adbc_con")) {
  tryCatch({
    adbcdrivermanager::adbc_connection_release(adbc_con)
    adbcdrivermanager::adbc_database_release(adbc_db)
  }, error = function(e) NULL)
  cat("ADBC connection closed.\n")
}
cat("Done.\n")